# Lab 09-01 · Crystals, Energy Bands and the Semiconductor

Companion to chapter [09-01](../../chapters/09-01-crystals-and-energy-bands.md). Handout: [README.md](README.md).

Everything here is built numerically from first principles: the diamond lattice from its FCC lattice and two-atom basis, and the Kronig–Penney bands from the **transfer matrix** of one period of the Schrödinger equation, not from the closed-form dispersion relation. Each part asserts that the numbers agree with the chapter.

In [1]:
import numpy as np
from itertools import product
from scipy.optimize import brentq
from scipy import constants as C
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HBAR, M0, Q = C.hbar, C.m_e, C.e
A_SI = 5.431e-8          # cm
print("constants loaded")

constants loaded


## Part 1 — The diamond lattice by counting

Generate every atom in a block of cubic cells (FCC points plus the basis offset $(a/4)(1,1,1)$), then count. Working in units of $a$ keeps coordinates exact.

**🔮 Predict:** how many atoms per cubic cell, how many nearest neighbors per atom, and at what distance?

In [2]:
def diamond(n=3):
    fcc = np.array([(0, 0, 0), (0, .5, .5), (.5, 0, .5), (.5, .5, 0)])
    basis = np.array([(0, 0, 0), (.25, .25, .25)])
    cells = np.array(list(product(range(n), repeat=3)))
    return (cells[:, None, None, :] + fcc[None, :, None, :] + basis[None, None, :, :]).reshape(-1, 3)

atoms = diamond(3)                         # 3 x 3 x 3 cells, units of a
print("atoms per cell:", len(atoms) / 27)
centre = np.array([1.0, 1.0, 1.0])        # an atom deep inside the block
d = np.linalg.norm(atoms - centre, axis=1)
shells = np.unique(np.round(d[d > 1e-9], 6))[:2]
counts = [int(np.sum(np.isclose(d, s))) for s in shells]
print("first shells (units of a):", shells, "counts:", counts)
N = len(atoms) / 27 / A_SI**3
print(f"atomic density {N:.4e} cm^-3, bond length {shells[0]*5.431:.3f} Å")
assert len(atoms) == 8 * 27 and counts[0] == 4 and counts[1] == 12
assert abs(N - 4.994e22) < 5e18

atoms per cell: 8.0
first shells (units of a): [0.433013 0.707107] counts: [4, 12]
atomic density 4.9940e+22 cm^-3, bond length 2.352 Å


### Planar densities

For a plane through the origin with normal $(hkl)$, choose two in-plane lattice vectors that tile the plane, count the atoms that lie in the plane inside the parallelogram they span, and divide by its area. That is an exact count, not an estimate.

In [3]:
def planar_density(normal, u, v, a=A_SI):
    normal, u, v = map(np.asarray, (normal, u, v))
    assert abs(normal @ u) < 1e-12 and abs(normal @ v) < 1e-12
    pts = diamond(6) - 3                      # a big block centred on the origin
    on = pts[np.isclose(pts @ normal, 0.0)]
    # coordinates in the (u, v) basis
    M = np.linalg.lstsq(np.stack([u, v], axis=1), on.T, rcond=None)[0].T
    inside = np.all((M > -1e-9) & (M < 1 - 1e-9), axis=1)
    area = np.linalg.norm(np.cross(u, v)) * a**2
    return inside.sum() / area

s100 = planar_density((1, 0, 0), (0, 1, 0), (0, 0, 1))
s110 = planar_density((1, 1, 0), (1, -1, 0), (0, 0, 1))
s111 = planar_density((1, 1, 1), (1, -1, 0), (0, 1, -1))
print(f"(100) {s100:.4e}   (110) {s110:.4e}   (111) {s111:.4e} cm^-2")
assert abs(s100 - 6.781e14) < 5e10 and abs(s110 - 9.589e14) < 5e10 and abs(s111 - 7.830e14) < 5e10

(100) 6.7806e+14   (110) 9.5893e+14   (111) 7.8296e+14 cm^-2


## Part 2 — Kronig–Penney bands from the transfer matrix

Over one period, the wavefunction and its slope, $(\psi, \psi')$, are carried forward by a 2 × 2 matrix $\mathbf{T}(E)$: free propagation over the well, then the barrier. Bloch's theorem says a traveling state exists only if the eigenvalues of $\mathbf{T}$ are $e^{\pm jkd}$, i.e. $\cos kd = \tfrac12\operatorname{tr}\mathbf{T}$. We build $\mathbf{T}$ by multiplying matrices numerically.

For a delta barrier of strength $g = V_0b$, $\psi$ is continuous and $\psi'$ jumps by $(2mg/\hbar^2)\psi = (2P/d)\psi$.

**🔮 Predict:** for $P$ = 3 and $d$ = 0.5 nm (Example 09-01.3), where is the bottom of the first band?

In [4]:
def free(alpha, L):
    c, s = np.cos(alpha * L), np.sin(alpha * L)
    return np.array([[c, s / alpha], [-alpha * s, c]])

def half_trace_delta(E_eV, P, d, m=M0):
    alpha = np.sqrt(2 * m * E_eV * Q) / HBAR
    kick = np.array([[1.0, 0.0], [2 * P / d, 1.0]])
    T = kick @ free(alpha, d)
    return 0.5 * np.trace(T)

P, d = 3.0, 0.5e-9
Ed = HBAR**2 / (2 * M0 * d**2) / Q
E = np.linspace(1e-4, 9 * np.pi**2 * Ed, 6000)
F = np.array([half_trace_delta(e, P, d) for e in E])
edges = [brentq(lambda e: half_trace_delta(e, P, d) - s, lo, hi)
         for s, lo, hi in [(1, 0.3, 0.9), (-1, 1.3, 1.6), (-1, 1.6, 4.0), (1, 5.0, 6.5)]]
print("Ed = %.4f eV" % Ed, " band edges (eV):", np.round(edges, 4))
assert abs(edges[0] - 0.5953) < 5e-4 and abs(edges[1] - 1.504) < 5e-4
assert abs(edges[2] - 2.883) < 5e-4 and abs(edges[3] - 6.016) < 5e-4

Ed = 0.1524 eV  band edges (eV): [0.5953 1.5041 2.8828 6.0165]


In [5]:
allowed = np.abs(F) <= 1
k = np.arccos(np.clip(F, -1, 1)) / d
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.6))
a1.plot(E, F); a1.axhline(1, ls="--", c="0.5"); a1.axhline(-1, ls="--", c="0.5")
a1.set_ylim(-2.5, 4); a1.set_xlabel("E (eV)"); a1.set_ylabel("½ tr T")
a2.plot(k[allowed] * d / np.pi, E[allowed], ".", ms=1.5)
a2.plot(-k[allowed] * d / np.pi, E[allowed], ".", ms=1.5)
a2.set_xlabel("kd/π"); a2.set_ylabel("E (eV)")
fig.tight_layout(); plt.close(fig)
print("allowed fraction of the energy axis:", allowed.mean().round(3))

allowed fraction of the energy axis: 0.734


### Effective mass by fitting

Near the band bottom, $E - E_b \approx \hbar^2k^2/2m^*$. Compute $k$ from the transfer matrix just above $E_b$ and fit.

In [6]:
Eb = edges[0]
dE = np.linspace(1e-5, 4e-4, 40)
ks = np.arccos([half_trace_delta(Eb + x, P, d) for x in dE]) / d
slope = np.polyfit(ks**2, dE * Q, 1)[0]          # J m^2
m_star = HBAR**2 / (2 * slope) / M0
print("m*/m0 =", round(m_star, 4))
assert abs(m_star - 1.125) < 2e-3

m*/m0 = 1.125


## Part 3 — From free electrons to isolated wells

Vary the barrier strength and track the first band's width, the first gap and the band-bottom mass.

**🔮 Predict (the chapter's guided exercise):** doubling $P$ from 3 to 6, does band 1 get wider or narrower? The gap? The mass? Write down a number for the new mass before running.

In [7]:
def first_band(P, d=0.5e-9):
    Ed = HBAR**2 / (2 * M0 * d**2) / Q
    f = lambda x: P * np.sin(x) / x + np.cos(x) if P > 0 else np.cos(x)
    xb = brentq(lambda x: f(x) - 1, 1e-6, np.pi - 1e-12)
    x2 = brentq(lambda x: f(x) + 1, np.pi + 1e-12, 2 * np.pi - 1e-9)
    # mass from the transfer matrix, as above
    Eb = Ed * xb**2
    dE = np.linspace(1e-6, 4e-5, 30) * Ed
    ks = np.arccos([half_trace_delta(Eb + x, P, d) for x in dE]) / d
    m = HBAR**2 / (2 * np.polyfit(ks**2, dE * Q, 1)[0]) / M0
    return Ed * (np.pi**2 - xb**2), Ed * (x2**2 - np.pi**2), m

rows = {P: first_band(P) for P in (0.01, 1, 3, 6, 12, 24)}
for P, (w, g, m) in rows.items():
    print(f"P = {P:5}: band 1 width {w:.3f} eV, first gap {g:.3f} eV, m*/m0 {m:.3f}")
w6, g6, m6 = rows[6]
assert abs(w6 - 0.6373) < 5e-4 and abs(g6 - 2.172) < 5e-4 and abs(m6 - 1.359) < 2e-3

P =  0.01: band 1 width 1.501 eV, first gap 0.006 eV, m*/m0 1.000
P =     1: band 1 width 1.244 eV, first gap 0.552 eV, m*/m0 1.019
P =     3: band 1 width 0.909 eV, first gap 1.379 eV, m*/m0 1.125
P =     6: band 1 width 0.637 eV, first gap 2.172 eV, m*/m0 1.359
P =    12: band 1 width 0.394 eV, first gap 2.994 eV, m*/m0 1.908
P =    24: band 1 width 0.222 eV, first gap 3.637 eV, m*/m0 3.086


## Part 4 — Silicon's masses and effective densities of states

Build the six valleys as inverse-mass tensors, average them for the conductivity mass, and add up their state counts for the density-of-states mass. Then compute $N_C$ by integrating the density of states against the Boltzmann factor, rather than from the closed form.

In [8]:
from scipy.integrate import quad
ml, mt, mhh, mlh = 0.98, 0.19, 0.49, 0.16
inv = []
for axis in range(3):
    m = np.full(3, mt); m[axis] = ml
    inv += [np.diag(1 / m)] * 2                    # +/- valleys on each axis
mc = 1 / np.mean([t[0, 0] for t in inv])
mdn = (sum(np.sqrt(np.prod(1 / np.diag(t))) for t in inv)) ** (2 / 3)
mdp = (mhh**1.5 + mlh**1.5) ** (2 / 3)
kT = C.k * 300
def N_eff(mr):
    g = lambda E: (1 / (2 * np.pi**2)) * (2 * mr * M0 / HBAR**2) ** 1.5 * np.sqrt(E)
    return quad(lambda x: g(x * kT) * np.exp(-x) * kT, 0, 60)[0] / 1e6   # cm^-3
print(f"m_c {mc:.4f}, m_dn {mdn:.4f}, m_dp {mdp:.4f}")
print(f"N_C {N_eff(mdn):.4e}, N_V {N_eff(mdp):.4e} cm^-3")
assert abs(mc - 0.2598) < 5e-5 and abs(mdn - 1.084) < 5e-4 and abs(mdp - 0.5492) < 5e-5
assert abs(N_eff(mdn) - 2.832e19) < 5e15 and abs(N_eff(mdp) - 1.021e19) < 5e15

m_c 0.2598, m_dn 1.0840, m_dp 0.5492
N_C 2.8320e+19, N_V 1.0213e+19 cm^-3


## Part 5 — Gaps with temperature and composition

Varshni's fit for Si and GaAs, and the Ga$_{1-y}$In$_y$As alloy: gap (with bowing 0.477 eV) and lattice constant (Vegard's law), with InP's lattice constant marked.

In [9]:
varshni = lambda T, E0, a, b: E0 - a * T**2 / (T + b)
T = np.linspace(4, 500, 300)
Eg_Si = varshni(T, 1.17, 4.73e-4, 636)
Eg_GaAs = varshni(T, 1.519, 5.405e-4, 204)
E_GaAs300 = varshni(300, 1.519, 5.405e-4, 204)
E_InAs300 = varshni(300, 0.417, 2.76e-4, 93)
y = np.linspace(0, 1, 201)
Eg_y = (1 - y) * E_GaAs300 + y * E_InAs300 - 0.477 * y * (1 - y)
a_y = (1 - y) * 5.65325 + y * 6.0583
y_match = brentq(lambda t: (1 - t) * 5.65325 + t * 6.0583 - 5.8697, 0, 1)
Eg_match = (1 - y_match) * E_GaAs300 + y_match * E_InAs300 - 0.477 * y_match * (1 - y_match)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.4))
a1.plot(T, Eg_Si, label="Si"); a1.plot(T, Eg_GaAs, label="GaAs"); a1.legend(); a1.set_xlabel("T (K)"); a1.set_ylabel("Eg (eV)")
a2.plot(y, Eg_y); a2.set_xlabel("In fraction y"); a2.set_ylabel("Eg at 300 K (eV)")
a3 = a2.twinx(); a3.plot(y, a_y, "C1"); a3.axhline(5.8697, ls=":", c="C1"); a3.set_ylabel("a (Å)")
fig.tight_layout(); plt.close(fig)
print(f"Si 300 K {varshni(300, 1.17, 4.73e-4, 636):.4f} eV; GaAs 300 K {E_GaAs300:.4f} eV")
print(f"lattice-matched y = {y_match:.4f}, gap {Eg_match:.4f} eV, cutoff {1239.84/Eg_match:.0f} nm")
# the matched alloy has y = 0.5344; the chapter's 0.7373 eV is for the rounded y = 0.53
Eg_053 = 0.47 * E_GaAs300 + 0.53 * E_InAs300 - 0.477 * 0.53 * 0.47
assert abs(y_match - 0.5344) < 5e-5 and abs(Eg_053 - 0.7373) < 5e-5

Si 300 K 1.1245 eV; GaAs 300 K 1.4225 eV
lattice-matched y = 0.5344, gap 0.7327 eV, cutoff 1692 nm


## Wrap-up

- Counting atoms in a generated lattice reproduces every density in Example 09-01.1, with no formula assumed.
- The transfer matrix of one period gives the Kronig–Penney bands of Example 09-01.3 directly; the closed-form relation is just its half-trace written out.
- Stronger barriers narrow the bands, widen the gaps and make carriers heavier; the mass grows slowly at first and roughly in proportion to $P$ for strong barriers (Problem 09-01.7).
- The valley masses reproduce silicon's $N_C$ and $N_V$ to within 2% of Hu's table.